In [ ]:
import pandas as pd
import numpy as np
import os

# ── Load raw data ──────────────────────────────────────────
matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

print("Matches shape:", matches.shape)
print("Deliveries shape:", deliveries.shape)
print("\nMissing values in matches:\n", matches.isnull().sum())

# ── Data Cleaning ──────────────────────────────────────────
# Fill missing values
matches['player_of_match'] = matches['player_of_match'].fillna('Unknown')
matches['city'] = matches['city'].fillna(matches['venue'].str.split(',').str[0])

# Drop duplicate rows if any
matches.drop_duplicates(inplace=True)
deliveries.drop_duplicates(inplace=True)

# Standardize team names (some teams were renamed over the years)
team_rename = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Kings XI Punjab': 'Punjab Kings'
}
matches['team1'] = matches['team1'].replace(team_rename)
matches['team2'] = matches['team2'].replace(team_rename)
matches['winner'] = matches['winner'].replace(team_rename)

# ── Feature Engineering ────────────────────────────────────
# 1. Toss advantage: did toss winner win the match?
matches['toss_win_match_win'] = (
    matches['toss_winner'] == matches['winner']
).astype(int)

# 2. Batting team total runs per match
batting_totals = deliveries.groupby(
    ['match_id', 'batting_team']
)['total_runs'].sum().reset_index()
batting_totals.columns = ['match_id', 'batting_team', 'total_runs']

# 3. Wickets per match per bowling team
wickets = deliveries[deliveries['player_dismissed'].notna()].groupby(
    ['match_id', 'bowling_team']
).size().reset_index(name='wickets_taken')

# 4. Merge all into a match-level feature table
match_features = matches[[
    'id', 'season', 'team1', 'team2', 'toss_winner',
    'toss_decision', 'winner', 'toss_win_match_win'
]].copy()

match_features['target'] = (
    match_features['winner'] == match_features['team1']
).astype(int)

# ── Save cleaned data ──────────────────────────────────────
os.makedirs('../data/processed', exist_ok=True)
matches.to_csv('../data/processed/matches_cleaned.csv', index=False)
deliveries.to_csv('../data/processed/deliveries_cleaned.csv', index=False)
match_features.to_csv('../data/processed/match_features.csv', index=False)

print("\nCleaned data saved to data/processed/")
print("Match features shape:", match_features.shape)